In [ ]:
if "spark" in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("postgres_silver_validation")
    .master("spark://spark-master:7077")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.autoBroadcastJoinThreshold", "10m")
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .config("spark.sql.legacy.parquet.nanosAsLong", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 16:30:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
POSTGRES_SILVER_PATH = (
    "s3a://end-to-end-streaming-data-platform-silver/"
    "postgres"
)

In [3]:
POSTGRES_BRONZE_PATH = (
    "s3a://end-to-end-streaming-data-platform-bronze/"
    "postgres/ingestion_date=2026-09-30-Sep"
)

POSTGRES_SILVER_PATH = (
    "s3a://end-to-end-streaming-data-platform-silver/"
    "postgres"
)

Read Bronze and Silver

In [4]:
df_users_bronze = spark.read.parquet(
    f"{POSTGRES_BRONZE_PATH}/users.parquet"
)

df_subscriptions_bronze = spark.read.parquet(
    f"{POSTGRES_BRONZE_PATH}/subscription.parquet"
)

df_payments_bronze = spark.read.parquet(
    f"{POSTGRES_BRONZE_PATH}/payments.parquet"
)

df_users_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/users.parquet"
)

df_subscriptions_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/subscription.parquet"
)

df_payments_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/payments.parquet"
)

print("All Bronze and Silver datasets loaded successfully.")

26/09/30 16:31:28 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

All Bronze and Silver datasets loaded successfully.


Dataset Counts

In [3]:
df_users_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/users.parquet"
)

df_subscriptions_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/subscription.parquet"
)

df_payments_silver = spark.read.parquet(
    f"{POSTGRES_SILVER_PATH}/payments.parquet"
)

26/09/06 23:57:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [5]:
bronze_counts = {
    "users": df_users_bronze.count(),
    "subscriptions": df_subscriptions_bronze.count(),
    "payments": df_payments_bronze.count(),
}

silver_counts = {
    "users": df_users_silver.count(),
    "subscriptions": df_subscriptions_silver.count(),
    "payments": df_payments_silver.count(),
}

print("Bronze counts:")
print(bronze_counts)

print("\nSilver counts:")
print(silver_counts)
print("=" * 70)

Bronze counts:
{'users': 10000, 'subscriptions': 20141, 'payments': 11767}

Silver counts:
{'users': 10000, 'subscriptions': 20141, 'payments': 11767}


Validate counts

In [6]:
assert silver_counts["users"] == bronze_counts["users"], \
    "Users row count changed."

assert silver_counts["subscriptions"] == bronze_counts["subscriptions"], \
    "Subscriptions row count changed."

assert silver_counts["payments"] == bronze_counts["payments"], \
    "Payments row count changed."

print("PASS — Row counts preserved.")
print("=" * 70)

PASS — Row counts preserved.


Expected Dataset Counts

In [7]:
assert silver_counts["users"] == 10000, \
    f"Unexpected users count: {silver_counts['users']}"

assert silver_counts["subscriptions"] == 20141, \
    f"Unexpected subscriptions count: {silver_counts['subscriptions']}"

assert silver_counts["payments"] == 11767, \
    f"Unexpected payments count: {silver_counts['payments']}"

print("PASS — Expected dataset counts confirmed.")
print("=" * 70)

PASS — Expected dataset counts confirmed.


Schema Validation

In [8]:
# users
expected_users_schema = {
    "user_id": "string",
    "email": "string",
    "country": "string",
    "account_status": "string",
    "created_at": "timestamp",
    "last_login": "timestamp",
    "is_missing_email": "boolean",
    "is_invalid_country": "boolean",
    "is_login_before_creation": "boolean",
    "created_date": "date",
    "last_login_date": "date",
}

actual_users_schema = {
    field.name: field.dataType.simpleString()
    for field in df_users_silver.schema.fields
}

assert actual_users_schema == expected_users_schema, (
    f"Users schema mismatch.\n"
    f"Expected: {expected_users_schema}\n"
    f"Actual: {actual_users_schema}"
)

print("PASS — Users schema.")

PASS — Users schema.


In [9]:
# subscription
expected_subscriptions_schema = {
    "subscription_id": "string",
    "user_id": "string",
    "plan_type": "string",
    "start_date": "timestamp",
    "end_date": "timestamp",
    "is_active": "boolean",
    "auto_renew": "boolean",
    "is_end_date_before_start_date": "boolean",
    "start_date_day": "date",
    "end_date_day": "date",
}

actual_subscriptions_schema = {
    field.name: field.dataType.simpleString()
    for field in df_subscriptions_silver.schema.fields
}

assert actual_subscriptions_schema == expected_subscriptions_schema, (
    f"Subscriptions schema mismatch.\n"
    f"Expected: {expected_subscriptions_schema}\n"
    f"Actual: {actual_subscriptions_schema}"
)

print("PASS — Subscriptions schema.")

PASS — Subscriptions schema.


In [10]:
# payments
expected_payments_schema = {
    "payment_id": "string",
    "subscription_id": "string",
    "amount": "decimal(10,2)",
    "currency": "string",
    "payment_date": "timestamp",
    "payment_status": "string",
    "payment_date_day": "date",
}

actual_payments_schema = {
    field.name: field.dataType.simpleString()
    for field in df_payments_silver.schema.fields
}

assert actual_payments_schema == expected_payments_schema, (
    f"Payments schema mismatch.\n"
    f"Expected: {expected_payments_schema}\n"
    f"Actual: {actual_payments_schema}"
)

print("PASS — Payments schema.")
print("=" * 70)

PASS — Payments schema.


Primary Key Validation

In [11]:
# users
duplicate_users = (
    df_users_silver
    .groupBy("user_id")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_users.count() == 0, \
    "Duplicate user_id values found."

assert df_users_silver.filter(
    F.col("user_id").isNull()
).count() == 0, \
    "Null user_id values found."

print("PASS — User IDs.")
print("=" * 70)

PASS — User IDs.


Subscriptions

In [12]:
duplicate_subscriptions = (
    df_subscriptions_silver
    .groupBy("subscription_id")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_subscriptions.count() == 0, \
    "Duplicate subscription_id values found."

assert df_subscriptions_silver.filter(
    F.col("subscription_id").isNull()
).count() == 0, \
    "Null subscription_id values found."

print("PASS — Subscription IDs.")
print("=" * 70)

PASS — Subscription IDs.


Payments

In [13]:
duplicate_payments = (
    df_payments_silver
    .groupBy("payment_id")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_payments.count() == 0, \
    "Duplicate payment_id values found."

assert df_payments_silver.filter(
    F.col("payment_id").isNull()
).count() == 0, \
    "Null payment_id values found."

print("PASS — Payment IDs.")

PASS — Payment IDs.


Relationship Validation

In [14]:
# Subscription → User
orphan_subscriptions = (
    df_subscriptions_silver
    .select("subscription_id", "user_id")
    .join(
        df_users_silver.select("user_id"),
        on="user_id",
        how="left_anti"
    )
)

assert orphan_subscriptions.count() == 0, \
    "Orphan subscriptions found."

print("PASS — Subscription → User relationship.")

PASS — Subscription → User relationship.


In [15]:
# Payment → Subscription
orphan_payments = (
    df_payments_silver
    .select("payment_id", "subscription_id")
    .join(
        df_subscriptions_silver.select("subscription_id"),
        on="subscription_id",
        how="left_anti"
    )
)

assert orphan_payments.count() == 0, \
    "Orphan payments found."

print("PASS — Payment → Subscription relationship.")

PASS — Payment → Subscription relationship.


Grain Validation

In [16]:
assert (
    df_users_silver.select("user_id").distinct().count()
    == df_users_silver.count()
), "Users grain changed."

assert (
    df_subscriptions_silver.select("subscription_id").distinct().count()
    == df_subscriptions_silver.count()
), "Subscriptions grain changed."

assert (
    df_payments_silver.select("payment_id").distinct().count()
    == df_payments_silver.count()
), "Payments grain changed."

print("PASS — Original grain preserved.")

PASS — Original grain preserved.


Dirty Data — Users

In [17]:
# Missing email
missing_email_count = (
    df_users_silver
    .filter(F.col("email").isNull())
    .count()
)

flagged_missing_email_count = (
    df_users_silver
    .filter(F.col("is_missing_email") == True)
    .count()
)

assert missing_email_count == flagged_missing_email_count, \
    "Missing email flag does not match actual missing emails."

assert missing_email_count == 1, \
    f"Expected 1 missing email, found {missing_email_count}"

print("PASS — Missing email detection.")

PASS — Missing email detection.


Invalid country

In [18]:
invalid_country_count = (
    df_users_silver
    .filter(F.col("country") == "XXX")
    .count()
)

flagged_invalid_country_count = (
    df_users_silver
    .filter(F.col("is_invalid_country") == True)
    .count()
)

assert invalid_country_count == 1, \
    f"Expected 1 XXX country, found {invalid_country_count}"

assert flagged_invalid_country_count == invalid_country_count, \
    "Invalid country flag mismatch."

print("PASS — Invalid country detection.")

PASS — Invalid country detection.


Login before creation

In [19]:
actual_login_before_creation = (
    df_users_silver
    .filter(
        F.col("last_login").isNotNull()
        & (F.col("last_login") < F.col("created_at"))
    )
    .count()
)

flagged_login_before_creation = (
    df_users_silver
    .filter(F.col("is_login_before_creation") == True)
    .count()
)

assert actual_login_before_creation == flagged_login_before_creation, \
    "Login chronology flag mismatch."

assert actual_login_before_creation == 1, \
    f"Expected 1 invalid login chronology, found {actual_login_before_creation}"

print("PASS — Login chronology detection.")

PASS — Login chronology detection.


Dirty Data — Subscriptions

In [20]:
actual_invalid_subscription_dates = (
    df_subscriptions_silver
    .filter(
        F.col("end_date").isNotNull()
        & (F.col("end_date") < F.col("start_date"))
    )
    .count()
)

flagged_invalid_subscription_dates = (
    df_subscriptions_silver
    .filter(
        F.col("is_end_date_before_start_date") == True
    )
    .count()
)

assert actual_invalid_subscription_dates == flagged_invalid_subscription_dates, \
    "Subscription chronology flag mismatch."

assert actual_invalid_subscription_dates == 1, \
    f"Expected 1 invalid subscription date, found {actual_invalid_subscription_dates}"

print("PASS — Subscription chronology detection.")

PASS — Subscription chronology detection.


Verify Dirty Values Were Preserved

In [21]:
# Invalid login preserved
assert (
    df_users_silver
    .filter(F.col("country") == "XXX")
    .count()
    == 1
)

print("PASS — Dirty country value preserved.")

PASS — Dirty country value preserved.


In [22]:
assert (
    df_users_silver
    .filter(
        (F.col("is_login_before_creation") == True)
        & F.col("last_login").isNotNull()
    )
    .count()
    == 1
)

print("PASS — Invalid login timestamp preserved.")

PASS — Invalid login timestamp preserved.


In [23]:
assert (
    df_subscriptions_silver
    .filter(
        (F.col("is_end_date_before_start_date") == True)
        & F.col("end_date").isNotNull()
    )
    .count()
    == 1
)

print("PASS — Invalid subscription end date preserved.")

PASS — Invalid subscription end date preserved.


In [24]:
print("Null values in user DQ flags:")

df_users_silver.select(
    F.sum(F.col("is_invalid_country").isNull().cast("int")).alias(
        "is_invalid_country_nulls"
    ),
    F.sum(F.col("is_login_before_creation").isNull().cast("int")).alias(
        "is_login_before_creation_nulls"
    ),
).show()

Null values in user DQ flags:
+------------------------+------------------------------+
|is_invalid_country_nulls|is_login_before_creation_nulls|
+------------------------+------------------------------+
|                       0|                             0|
+------------------------+------------------------------+



Technical Transformation Validation

In [25]:
email_with_outer_spaces = (
    df_users_silver
    .filter(
        F.col("email").isNotNull()
        & (F.col("email") != F.trim(F.col("email")))
    )
    .count()
)

assert email_with_outer_spaces == 0, \
    "Email values still contain leading/trailing spaces."

print("PASS — Email trimming.")

PASS — Email trimming.


In [26]:
# Country normalization
country_not_normalized = (
    df_users_silver
    .filter(
        F.col("country").isNotNull()
        & (
            (F.col("country") != F.upper(F.col("country")))
            | (F.col("country") != F.trim(F.col("country")))
        )
    )
    .count()
)

assert country_not_normalized == 0, \
    "Country values are not technically normalized."

print("PASS — Country normalization.")

PASS — Country normalization.


In [27]:
currency_not_normalized = (
    df_payments_silver
    .filter(
        F.col("currency").isNotNull()
        & (
            (F.col("currency") != F.upper(F.col("currency")))
            | (F.col("currency") != F.trim(F.col("currency")))
        )
    )
    .count()
)

assert currency_not_normalized == 0, \
    "Currency values are not normalized."

print("PASS — Currency normalization.")

PASS — Currency normalization.


In [28]:
status_not_normalized = (
    df_payments_silver
    .filter(
        F.col("payment_status").isNotNull()
        & (
            (F.col("payment_status") != F.lower(F.col("payment_status")))
            | (F.col("payment_status") != F.trim(F.col("payment_status")))
        )
    )
    .count()
)

assert status_not_normalized == 0, \
    "Payment status values are not normalized."

print("PASS — Payment status normalization.")

PASS — Payment status normalization.


Date Derivation Validation

In [29]:
invalid_created_date = (
    df_users_silver
    .filter(
        F.col("created_date")
        != F.to_date(F.col("created_at"))
    )
    .count()
)

assert invalid_created_date == 0, \
    "created_date derivation mismatch."

invalid_last_login_date = (
    df_users_silver
    .filter(
        F.col("last_login_date")
        != F.to_date(F.col("last_login"))
    )
    .count()
)

assert invalid_last_login_date == 0, \
    "last_login_date derivation mismatch."

print("PASS — User date derivations.")

PASS — User date derivations.


In [30]:
invalid_start_date = (
    df_subscriptions_silver
    .filter(
        F.col("start_date_day")
        != F.to_date(F.col("start_date"))
    )
    .count()
)

invalid_end_date = (
    df_subscriptions_silver
    .filter(
        F.col("end_date_day")
        != F.to_date(F.col("end_date"))
    )
    .count()
)

assert invalid_start_date == 0, \
    "start_date_day derivation mismatch."

assert invalid_end_date == 0, \
    "end_date_day derivation mismatch."

print("PASS — Subscription date derivations.")

PASS — Subscription date derivations.


In [31]:
invalid_payment_date = (
    df_payments_silver
    .filter(
        F.col("payment_date_day")
        != F.to_date(F.col("payment_date"))
    )
    .count()
)

assert invalid_payment_date == 0, \
    "payment_date_day derivation mismatch."

print("PASS — Payment date derivation.")

PASS — Payment date derivation.


Timestamp Preservation

In [32]:
users_timestamp_check = (
    df_users_bronze.alias("b")
    .join(
        df_users_silver.alias("s"),
        on="user_id",
        how="inner"
    )
    .filter(
        F.col("b.created_at") != F.col("s.created_at")
    )
    .count()
)

assert users_timestamp_check == 0, \
    "created_at changed during transformation."

print("PASS — User created_at preserved.")

PASS — User created_at preserved.


In [33]:
subscriptions_timestamp_check = (
    df_subscriptions_bronze.alias("b")
    .join(
        df_subscriptions_silver.alias("s"),
        on="subscription_id",
        how="inner"
    )
    .filter(
        (F.col("b.start_date") != F.col("s.start_date"))
        | (
            F.col("b.end_date").isNotNull()
            & F.col("s.end_date").isNotNull()
            & (F.col("b.end_date") != F.col("s.end_date"))
        )
    )
    .count()
)

assert subscriptions_timestamp_check == 0, \
    "Subscription timestamps changed."

print("PASS — Subscription timestamps preserved.")

PASS — Subscription timestamps preserved.


In [34]:
payments_timestamp_check = (
    df_payments_bronze.alias("b")
    .join(
        df_payments_silver.alias("s"),
        on="payment_id",
        how="inner"
    )
    .filter(
        F.col("b.payment_date") != F.col("s.payment_date")
    )
    .count()
)

assert payments_timestamp_check == 0, \
    "Payment timestamps changed."

print("PASS — Payment timestamps preserved.")

PASS — Payment timestamps preserved.


Amount Validation

In [35]:
amount_check = (
    df_payments_bronze.alias("b")
    .join(
        df_payments_silver.alias("s"),
        on="payment_id",
        how="inner"
    )
    .filter(
        F.col("b.amount").cast("decimal(10,2)")
        != F.col("s.amount")
    )
    .count()
)

assert amount_check == 0, \
    "Payment amount changed during transformation."

print("PASS — Payment amounts preserved.")

PASS — Payment amounts preserved.


Flag Consistency

In [36]:
users_flag_consistency = (
    df_users_silver
    .filter(
        (
            (F.col("is_missing_email") == True)
            != F.col("email").isNull()
        )
        |
        (
            (F.col("is_login_before_creation") == True)
            != (
                F.col("last_login").isNotNull()
                & (F.col("last_login") < F.col("created_at"))
            )
        )
    )
    .count()
)

assert users_flag_consistency == 0, \
    "User DQ flags are inconsistent with the underlying data."

print("PASS — User flag consistency.")

PASS — User flag consistency.


In [37]:
VALID_COUNTRIES = (
    "KSA", "EGY", "UAE", "QAT",
    "KWT", "JOR", "MAR"
)

invalid_country_flag_check = (
    df_users_silver
    .filter(
        (
            F.col("is_invalid_country") == True
        )
        != (
            F.col("country").isNull()
            | ~F.col("country").isin(*VALID_COUNTRIES)
        )
    )
    .count()
)

assert invalid_country_flag_check == 0, \
    "Invalid country flags are inconsistent."

print("PASS — Country flag consistency.")

PASS — Country flag consistency.


In [38]:
subscription_flag_check = (
    df_subscriptions_silver
    .filter(
        (
            F.col("is_end_date_before_start_date") == True
        )
        != (
            F.col("end_date").isNotNull()
            & (F.col("end_date") < F.col("start_date"))
        )
    )
    .count()
)

assert subscription_flag_check == 0, \
    "Subscription DQ flags are inconsistent."

print("PASS — Subscription flag consistency.")

PASS — Subscription flag consistency.


Final Validation Summary

In [39]:
print("=" * 60)
print("POSTGRESQL SILVER VALIDATION SUMMARY")
print("=" * 60)

print("\nDataset counts:")
print(f"Users:          {silver_counts['users']}")
print(f"Subscriptions:  {silver_counts['subscriptions']}")
print(f"Payments:       {silver_counts['payments']}")

print("\nExpected dirty records:")
print(f"Missing emails:              {missing_email_count}")
print(f"Invalid countries:           {invalid_country_count}")
print(f"Login before creation:       {actual_login_before_creation}")
print(f"Invalid subscription dates:  {actual_invalid_subscription_dates}")

print("\nRelationship checks:")
print("Orphan subscriptions: 0")
print("Orphan payments:       0")

print("\nValidation status:")
print("PASS — All PostgreSQL Silver validation checks completed successfully.")

POSTGRESQL SILVER VALIDATION SUMMARY

Dataset counts:
Users:          10000
Subscriptions:  20141
Payments:       11767

Expected dirty records:
Missing emails:              1
Invalid countries:           1
Login before creation:       1
Invalid subscription dates:  1

Relationship checks:
Orphan subscriptions: 0
Orphan payments:       0

Validation status:
PASS — All PostgreSQL Silver validation checks completed successfully.


Optional final hard gate

In [40]:
SILVER_VALIDATION_PASSED = True

print("======================================")
print("POSTGRESQL SILVER VALIDATION: PASSED")
print("======================================")

POSTGRESQL SILVER VALIDATION: PASSED


In [42]:
df_users_silver.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_login: timestamp (nullable = true)
 |-- is_missing_email: boolean (nullable = true)
 |-- is_invalid_country: boolean (nullable = true)
 |-- is_login_before_creation: boolean (nullable = true)
 |-- created_date: date (nullable = true)
 |-- last_login_date: date (nullable = true)



In [41]:
gold_dim_users = (
    df_users_silver
    .filter(
        (F.col("is_missing_email") == False) & 
        (F.col("is_invalid_country") == False) & 
        (F.col("is_login_before_creation") == False)
    )
    .select(
        F.col("user_id"),
        F.col("email"),
        F.col("country"),
        F.col("account_status"),
        F.col("created_date").alias("created_at"),
        F.col("last_login_date").alias("last_login")
    )
)

In [42]:
gold_dim_users \
    .write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-gold/dim_users/")

In [43]:
df_dim_users = spark.read.parquet(
    "s3a://end-to-end-streaming-data-platform-gold/dim_users/"
)

In [44]:
df_dim_users.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: date (nullable = true)
 |-- last_login: date (nullable = true)



In [45]:
print(
    "Users:",
    df_dim_users.count(),
)

Users: 9997


In [13]:
df_subscriptions_silver.printSchema()

root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)
 |-- is_end_date_before_start_date: boolean (nullable = true)
 |-- start_date_day: date (nullable = true)
 |-- end_date_day: date (nullable = true)



In [46]:
gold_fact_subscription = (
    df_subscriptions_silver
    
    .filter(F.col("is_end_date_before_start_date") == False)
    
    .select(
        F.col("subscription_id"),
        F.col("user_id"),
        F.col("plan_type"),
        F.col("is_active"),
        F.col("auto_renew"),
        F.col("start_date_day").alias("start_date"), 
        F.col("end_date_day").alias("end_date")       
    )
)

In [47]:
gold_fact_subscription \
    .write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-gold/fact_subscription/")

In [48]:
df_fact_subscription = spark.read.parquet(
    "s3a://end-to-end-streaming-data-platform-gold/fact_subscription/"
)

In [49]:
df_fact_subscription.printSchema()

root
 |-- subscription_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- plan_type: string (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- auto_renew: boolean (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)



In [50]:
print(
    "subscription:",
    df_fact_subscription.count(),
)

subscription: 20140


In [19]:
df_payments_silver.printSchema()

root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_date_day: date (nullable = true)



In [51]:
gold_fact_payments = (
    df_payments_silver              
    .select(
        F.col("payment_id"),
        F.col("subscription_id"),
        F.col("amount"),
        F.col("currency"),
        F.col("payment_status"),
        F.col("payment_date_day").alias("payment_date") 
    )
)

In [52]:
gold_fact_payments \
    .write \
    .mode("overwrite") \
    .parquet("s3a://end-to-end-streaming-data-platform-gold/fact_payments/")

In [53]:
df_fact_payments = spark.read.parquet(
    "s3a://end-to-end-streaming-data-platform-gold/fact_payments/"
)

In [54]:
df_fact_payments.printSchema()

root
 |-- payment_id: string (nullable = true)
 |-- subscription_id: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- currency: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_date: date (nullable = true)



In [55]:
df_fact_payments.count()

11767